In [1]:
from helpers import *
import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

In [2]:
def aggregate_gpu_data_new(year: str) -> pd.DataFrame:
    """
    Aggregates GPU usage data for all months in a given year by merging job records with node statistics.

    Parameters:
        year (str): Two-digit year string (e.g., "25" for 2025).

    Returns:
        pd.DataFrame: A merged DataFrame containing job and GPU usage records for the entire year.
    """
    # Validate year format
    if not isinstance(year, str) or not year.isdigit() or len(year) != 2:
        raise ValueError(
            f"Invalid year format: {year}. Expected a two-digit string (e.g., '25' for 2025)."
        )

    # Load job records for the entire year
    gpu_jobs = read_gpu_records(f"/projectnb/rcsmetrics/accounting/data/scc/20{year}.csv")
    gpu_jobs["task_string"] = gpu_jobs["task_number"].astype(str)
    gpu_jobs.loc[~(gpu_jobs["options"].str.contains("-t")), "task_string"] = "undefined"
    gpu_jobs["job_task"] = (
        gpu_jobs["job_number"].astype(str) + "." + gpu_jobs["task_string"].astype(str)
    )

    # Get all files for the entire year
    nodes = os.listdir("/project/scv/dugan/gpustats/data/")
    files = []
    for month in [f"{m:02d}" for m in range(1, 13)]:  # Generate "01" to "12"
        files.extend([
            f"/project/scv/dugan/gpustats/data/{node}/{year}{month}"
            for node in nodes
            if os.path.exists(f"/project/scv/dugan/gpustats/data/{node}/{year}{month}")
        ])

    # Process and merge data
    all_merged_dfs = []
    for file_name in files:
        try:
            gpu_records = pd.DataFrame(clean_gpu_data(file_name))
        except Exception as e:
            print(f"Skipping missing or corrupted file: {file_name}")
            continue

        # Merge with job records
        merged_df = pd.merge(
            gpu_records, gpu_jobs, left_on="job_id", right_on="job_task", how="left"
        )
        all_merged_dfs.append(merged_df)

    # Return the final concatenated DataFrame
    return pd.concat(all_merged_dfs, ignore_index=True) if all_merged_dfs else pd.DataFrame()

In [3]:
year_test = aggregate_gpu_data_new('24')
year_test

,time,bus,util,memory_throughput,user,project_x,job_id,scenario,qname,hostname,...,granted_pe,slots,task_number,cpu,options,pe_taskid,maxvmem,n_gpu,task_string,job_task
0,1704085201,00000000:17:00.0,90.0,72.0,zjguo,depend,3367552.undefined,1,ece,scc-216,...,omp8,8.0,0.0,93886.545949,"-U engineering,ece -u zjguo -l gpu_compute_cap...",NaN,1.175716e+11,1.0,undefined,3367552.undefined
1,1704085201,00000000:65:00.0,22.0,1.0,dgordon,textconv,3366341.undefined,1,ece,scc-216,...,omp,4.0,0.0,322113.971996,"-U ece,engineering -u dgordon -l gpu_compute_c...",NaN,3.344766e+11,1.0,undefined,3366341.undefined
2,1704085201,00000000:CA:00.0,83.0,52.0,animikh,rlvn,3321766.undefined,1,ece,scc-216,...,omp8,8.0,0.0,689953.903543,"-U rlvn,ece,academic,engineering -u animikh -l...",NaN,5.860159e+11,2.0,undefined,3321766.undefined
3,1704085201,00000000:E3:00.0,80.0,46.0,animikh,rlvn,3321766.undefined,1,ece,scc-216,...,omp8,8.0,0.0,689953.903543,"-U rlvn,ece,academic,engineering -u animikh -l...",NaN,5.860159e+11,2.0,undefined,3321766.undefined
4,1704085501,00000000:17:00.0,92.0,75.0,zjguo,depend,3367552.undefined,1,ece,scc-216,...,omp8,8.0,0.0,93886.545949,"-U engineering,ece -u zjguo -l gpu_compute_cap...",NaN,1.175716e+11,1.0,undefined,3367552.undefined
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38379462,1735706701,00000000:82:00.0,0.0,0.0,-,-,-,0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
38379463,1735707001,00000000:02:00.0,0.0,0.0,-,-,-,0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
38379464,1735707001,00000000:82:00.0,0.0,0.0,-,-,-,0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
38379465,1735707301,00000000:02:00.0,0.0,0.0,-,-,-,0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
year_test_old = aggregate_gpu_data('24')


Processing 24-01...
Processing 24-02...
Processing 24-03...
Processing 24-04...
Processing 24-05...
Processing 24-06...
Processing 24-07...
Processing 24-08...
Processing 24-09...
Processing 24-10...
Processing 24-11...
Processing 24-12...


In [5]:
year_test.equals(year_test_old)

True

In [6]:
jan = clean_gpu_data(f"/project/scv/dugan/gpustats/data/scc-x05/2401")
feb = clean_gpu_data(f"/project/scv/dugan/gpustats/data/scc-x05/2402")
jan.tail(10)

,time,bus,util,memory_throughput,user,project,job_id,scenario
17622,1706762101,00000000:3B:00.0,91.0,44.0,dgooding,snoplus,4198041.undefined,1
17623,1706762101,00000000:D8:00.0,0.0,0.0,-,-,-,0
17624,1706762401,00000000:3B:00.0,89.0,68.0,dgooding,snoplus,4198041.undefined,1
17625,1706762401,00000000:D8:00.0,48.0,9.0,gvfranco,mcnet,4197929.177,1
17626,1706762701,00000000:3B:00.0,89.0,70.0,dgooding,snoplus,4198041.undefined,1
17627,1706762701,00000000:D8:00.0,0.0,0.0,-,-,-,0
17628,1706763001,00000000:3B:00.0,89.0,42.0,dgooding,snoplus,4198041.undefined,1
17629,1706763001,00000000:D8:00.0,52.0,15.0,gvfranco,mcnet,4197929.206,1
17630,1706763301,00000000:3B:00.0,100.0,81.0,dgooding,snoplus,4198041.undefined,1
17631,1706763301,00000000:D8:00.0,51.0,15.0,gvfranco,mcnet,4197929.230,1


In [7]:
feb.head(10)

,time,bus,util,memory_throughput,user,project,job_id,scenario
0,1706763601,00000000:3B:00.0,89.0,49.0,dgooding,snoplus,4198041.undefined,1
1,1706763601,00000000:D8:00.0,53.0,15.0,gvfranco,mcnet,4197929.230,1
2,1706763901,00000000:3B:00.0,90.0,74.0,dgooding,snoplus,4198041.undefined,1
3,1706763901,00000000:D8:00.0,51.0,16.0,gvfranco,mcnet,4197929.230,1
4,1706764201,00000000:3B:00.0,99.0,63.0,dgooding,snoplus,4198041.undefined,1
5,1706764201,00000000:D8:00.0,56.0,18.0,gvfranco,mcnet,4197929.230,1
6,1706764501,00000000:3B:00.0,89.0,38.0,dgooding,snoplus,4198041.undefined,1
7,1706764501,00000000:D8:00.0,53.0,15.0,gvfranco,mcnet,4197929.230,1
8,1706764801,00000000:3B:00.0,89.0,50.0,dgooding,snoplus,4198041.undefined,1
9,1706764801,00000000:D8:00.0,55.0,16.0,gvfranco,mcnet,4197929.230,1


In [8]:
year_test_old['job_id'].str.contains('4197929.230').any()

np.True_

In [9]:
year_test_old[year_test_old['job_id']=='4197929.230']

,time,bus,util,memory_throughput,user,project_x,job_id,scenario,qname,hostname,...,granted_pe,slots,task_number,cpu,options,pe_taskid,maxvmem,n_gpu,task_string,job_task
2222533,1706763301,00000000:D8:00.0,51.0,15.0,gvfranco,mcnet,4197929.230,1,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4722561,1706763601,00000000:D8:00.0,53.0,15.0,gvfranco,mcnet,4197929.230,1,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4722563,1706763901,00000000:D8:00.0,51.0,16.0,gvfranco,mcnet,4197929.230,1,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4722565,1706764201,00000000:D8:00.0,56.0,18.0,gvfranco,mcnet,4197929.230,1,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4722567,1706764501,00000000:D8:00.0,53.0,15.0,gvfranco,mcnet,4197929.230,1,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4722569,1706764801,00000000:D8:00.0,55.0,16.0,gvfranco,mcnet,4197929.230,1,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4722571,1706765101,00000000:D8:00.0,52.0,12.0,gvfranco,mcnet,4197929.230,1,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
year_test_old['job_id'].str.contains('4198041.undefined').any()

np.True_

In [11]:
year_test_old[year_test_old['job_id']=='4198041.undefined']

,time,bus,util,memory_throughput,user,project_x,job_id,scenario,qname,hostname,...,granted_pe,slots,task_number,cpu,options,pe_taskid,maxvmem,n_gpu,task_string,job_task
2222490,1706757001,00000000:3B:00.0,0.0,0.0,Missing Values,Missing Values,4198041.undefined,2,v100,scc-x05,...,omp,4.0,0.0,24926.037819,"-u dgooding -l gpu_compute_capability=6.0,gpu_...",NaN,2.635636e+10,1.0,undefined,4198041.undefined
2222492,1706757301,00000000:3B:00.0,0.0,0.0,Missing Values,Missing Values,4198041.undefined,2,v100,scc-x05,...,omp,4.0,0.0,24926.037819,"-u dgooding -l gpu_compute_capability=6.0,gpu_...",NaN,2.635636e+10,1.0,undefined,4198041.undefined
2222494,1706757601,00000000:3B:00.0,86.0,39.0,dgooding,snoplus,4198041.undefined,1,v100,scc-x05,...,omp,4.0,0.0,24926.037819,"-u dgooding -l gpu_compute_capability=6.0,gpu_...",NaN,2.635636e+10,1.0,undefined,4198041.undefined
2222496,1706757901,00000000:3B:00.0,100.0,81.0,dgooding,snoplus,4198041.undefined,1,v100,scc-x05,...,omp,4.0,0.0,24926.037819,"-u dgooding -l gpu_compute_capability=6.0,gpu_...",NaN,2.635636e+10,1.0,undefined,4198041.undefined
2222498,1706758201,00000000:3B:00.0,88.0,40.0,dgooding,snoplus,4198041.undefined,1,v100,scc-x05,...,omp,4.0,0.0,24926.037819,"-u dgooding -l gpu_compute_capability=6.0,gpu_...",NaN,2.635636e+10,1.0,undefined,4198041.undefined
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4722674,1706780701,00000000:3B:00.0,92.0,41.0,dgooding,snoplus,4198041.undefined,1,v100,scc-x05,...,omp,4.0,0.0,24926.037819,"-u dgooding -l gpu_compute_capability=6.0,gpu_...",NaN,2.635636e+10,1.0,undefined,4198041.undefined
4722676,1706781001,00000000:3B:00.0,79.0,36.0,dgooding,snoplus,4198041.undefined,1,v100,scc-x05,...,omp,4.0,0.0,24926.037819,"-u dgooding -l gpu_compute_capability=6.0,gpu_...",NaN,2.635636e+10,1.0,undefined,4198041.undefined
4722678,1706781301,00000000:3B:00.0,88.0,67.0,dgooding,snoplus,4198041.undefined,1,v100,scc-x05,...,omp,4.0,0.0,24926.037819,"-u dgooding -l gpu_compute_capability=6.0,gpu_...",NaN,2.635636e+10,1.0,undefined,4198041.undefined
4722680,1706781602,00000000:3B:00.0,87.0,39.0,dgooding,snoplus,4198041.undefined,1,v100,scc-x05,...,omp,4.0,0.0,24926.037819,"-u dgooding -l gpu_compute_capability=6.0,gpu_...",NaN,2.635636e+10,1.0,undefined,4198041.undefined


## Iterating over files rather than monthly is equal but slower, cross month jobs are still present

In [12]:
# test process_gpu_data_range(start_date: str, end_date: str)

ranged = process_gpu_data_range("2024-01-01","2024-12-31")
ranged.equals(year_test_old)

Processing 24-01...
Processing 24-02...
Processing 24-03...
Processing 24-04...
Processing 24-05...
Processing 24-06...
Processing 24-07...
Processing 24-08...
Processing 24-09...
Processing 24-10...
Processing 24-11...
Processing 24-12...


True

In [ ]:
# check jobs that cross months for monthly, or jobs that cross years for yearly